# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Source: *The State of AI-Driven SEO — FlyRank Data Report, April 2026* (`state-of-seo-2026.flyrank.ai`), the paper this week's live session walked through.

### Finding 1 — "Which Pages Will Grow?" (growth prediction model)

**The claim, in my own words:** FlyRank trained a model to predict whether a page is growing or declining. They report it gets this right about 90% of the time on new pages from brands already in its training data, but only about 75% on brands it has never seen before, and they present both numbers side by side as the honest range.

**Where does the label come from?** It's an observed outcome, not a hand-defined rule — pages sorted into a growing or declining group off of real trend data, the same general shape as this project's own `is_declining_proxy` (a percent-change threshold on real impressions, not an opinion invented after the fact).

**Does the validation design carry the claim?** Partly, and the paper's own reporting is what lets me say so. The gap between "same brand, new pages" (90%) and "unseen brands" (75%) is exactly this notebook's Section 2 before/after logic — same-brand testing is the naive split (a brand's pages can share hidden character across train and test), unseen-brand testing is the grouped split. To their credit, they report both instead of only the flattering one. What's missing is the base rate: neither number is shown next to the class balance of growing vs. declining pages in the test set, so it's not possible to tell from the paper alone how much of that 75%/90% is real skill versus a skewed label distribution — the same check `building-baselines` insists on before trusting any precision number.

**Methodology question I'd ask them:** What is the growing/declining base rate in each held-out test set, and does the 75% unseen-brand number still beat it by a real margin — or does it partly reflect how imbalanced the label is?

### Finding 2 — "Refreshing Pages Actually Works" (refresh ROI)

**The claim, in my own words:** Comparing refreshed pages against stale ones, the paper reports large impression lifts in several age-and-competition segments (for example, a very large lift for old, high-competition pages that got refreshed), with most segments passing a statistical significance test on held-out data.

**Where does the label come from?** This is where it gets murkier than Finding 1. "Refreshed" vs. "stale" isn't an outcome that happened to a page independent of anyone's choice — it's the result of an editor or workflow *deciding* to refresh that specific page. The paper's own limitations note that content age confounds these comparisons, which is an admission of exactly this problem.

**Does the validation design carry the claim?** The significance testing (Kruskal-Wallis / Mann-Whitney, confidence intervals) shows the *difference* between refreshed and stale groups is real and not noise. It does not show the difference is *caused* by the refresh itself. If editors tend to refresh pages that already looked promising — decent prior traffic, a topic worth the effort — then "refreshed pages do better" could partly just be "editors are good at picking pages that were already going to do better," which is precisely the `hunting-leakage-and-validating` skill's warning about decision-derived signals: a flag produced by an existing system (here, an editor's judgment about which pages deserve a refresh) can look like a strong predictor while mostly just encoding that judgment, not the world.

**Methodology question I'd ask them:** Were the refreshed pages selected at random from eligible candidates, or did an editor/workflow choose them — and if the latter, was a matched or randomized comparison group ever tested to separate "the refresh helped" from "good pages get refreshed first"?

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### 2.0 Setup — rebuild the exact March feature frame and model matrix from Week 5

In [14]:
%pip install -q duckdb scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [15]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Paste your Hugging Face READ token (hf_...): "
)

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
SNAPSHOT = "2026-03-31"
MONTH = "2026-03"
PREV_MONTH = "2026-02"

print("Connected to Hugging Face warehouse")
print(f"Snapshot date    : {SNAPSHOT}")
print(f"Feature month    : {MONTH}")
print(f"Comparison month : {PREV_MONTH}")

Connected to Hugging Face warehouse
Snapshot date    : 2026-03-31
Feature month    : 2026-03
Comparison month : 2026-02


In [16]:
# Identical rebuild to w05_model.ipynb Section 0 -- same tables, same joins, same proxy label.
con.execute(f"""
    CREATE OR REPLACE TABLE march_agg AS
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions)  AS gsc_impressions_mar,
        SUM(gsc_clicks)       AS gsc_clicks_mar,
        AVG(gsc_avg_position) AS avg_position_mar,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_pageviews ELSE NULL END)        AS ga4_pageviews_mar,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE NULL END) AS ga4_engaged_sessions_mar
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""")

con.execute(f"""
    CREATE OR REPLACE TABLE feb_agg AS
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_feb
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={PREV_MONTH}/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""")

con.execute(f"""
    CREATE OR REPLACE TABLE content_meta AS
    SELECT content_hash_id, client_hash_id, content_type, word_count, backlinks,
           content_created_date, content_updated_date,
           last_optimized_date, optimization_eligible_date
    FROM read_parquet('{BASE}/dim_content.parquet')
    WHERE is_published IS TRUE AND is_deleted IS FALSE
""")

print('march_agg, feb_agg, content_meta rebuilt.')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

march_agg, feb_agg, content_meta rebuilt.


In [17]:
feature_frame = con.sql(f"""
    SELECT
        m.client_hash_id,
        m.content_hash_id,
        m.gsc_impressions_mar,
        m.gsc_clicks_mar,
        m.avg_position_mar,
        m.ga4_pageviews_mar,
        m.ga4_engaged_sessions_mar,
        c.content_created_date,
        c.content_updated_date,
        c.word_count,
        c.content_type,
        c.backlinks,
        f.gsc_impressions_feb,
        CASE
            WHEN f.gsc_impressions_feb > 0
             AND (m.gsc_impressions_mar - f.gsc_impressions_feb) * 1.0 / f.gsc_impressions_feb <= -0.20
            THEN 1 ELSE 0
        END AS is_declining_proxy
    FROM march_agg m
    JOIN feb_agg f USING (client_hash_id, content_hash_id)
    JOIN content_meta c USING (client_hash_id, content_hash_id)
    WHERE f.gsc_impressions_feb > 0
""").df()

feature_frame['ctr_mar'] = (
    feature_frame['gsc_clicks_mar'] / feature_frame['gsc_impressions_mar'].replace(0, np.nan)
)
feature_frame['days_since_update'] = (
    pd.Timestamp(SNAPSHOT) - pd.to_datetime(feature_frame['content_updated_date'])
).dt.days.clip(lower=0)
feature_frame['content_age_days'] = (
    pd.Timestamp(SNAPSHOT) - pd.to_datetime(feature_frame['content_created_date'])
).dt.days

print(f'Feature frame rows: {len(feature_frame)}')

Feature frame rows: 134086


In [18]:
# Same feature matrix as w05_model.ipynb Section 3.1 -- unchanged, so the split is the ONLY variable.
feature_frame['has_ga4_mar'] = (
    feature_frame['ga4_pageviews_mar'].notna() & (feature_frame['ga4_pageviews_mar'] > 0)
)
feature_frame['ga4_engagement_rate_mar'] = np.where(
    feature_frame['has_ga4_mar'],
    feature_frame['ga4_engaged_sessions_mar'] / feature_frame['ga4_pageviews_mar'].replace(0, np.nan),
    0.0,
)
feature_frame['ga4_engagement_rate_mar'] = feature_frame['ga4_engagement_rate_mar'].fillna(0.0)

feature_frame['has_word_count'] = feature_frame['word_count'].notna()
feature_frame['word_count_filled'] = feature_frame['word_count'].fillna(feature_frame['word_count'].median())
feature_frame['backlinks_filled'] = feature_frame['backlinks'].fillna(0)
feature_frame['log_impressions_mar'] = np.log1p(feature_frame['gsc_impressions_mar'])

content_type_dummies = pd.get_dummies(feature_frame['content_type'], prefix='ctype', dummy_na=True)

NUMERIC_FEATURES = [
    'log_impressions_mar', 'ctr_mar', 'avg_position_mar', 'content_age_days',
    'days_since_update', 'word_count_filled', 'backlinks_filled',
    'ga4_engagement_rate_mar', 'has_ga4_mar', 'has_word_count',
]

X = pd.concat(
    [feature_frame[NUMERIC_FEATURES].astype(float), content_type_dummies.astype(float)], axis=1
).fillna(0.0)
y = feature_frame['is_declining_proxy'].values
groups = feature_frame['client_hash_id'].values

print(f'Model matrix: {X.shape[0]} rows x {X.shape[1]} columns')

Model matrix: 134086 rows x 14 columns


### 2.1 Before — a naive split (no grouping)

`KFold` with `shuffle=True` scatters a single client's pages across both train and test at random. If clients have their own baseline "personality" (a consistently high- or low-traffic account, a house style, a niche with its own typical CTR), the model can partly memorize *which client this is* rather than learn a signal that generalizes to a client it has never seen — exactly the "rows from one group share hidden character" warning from `hunting-leakage-and-validating`.

In [19]:
from sklearn.model_selection import KFold, GroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

logreg = Pipeline([
    ('scale', StandardScaler()),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced')),
])
rf = RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=20,
    class_weight='balanced', random_state=42, n_jobs=-1,
)

naive_kfold = KFold(n_splits=5, shuffle=True, random_state=42)

oof_logreg_naive = cross_val_predict(logreg, X, y, cv=naive_kfold, method='predict_proba')[:, 1]
oof_rf_naive = cross_val_predict(rf, X, y, cv=naive_kfold, method='predict_proba')[:, 1]

auc_logreg_naive = roc_auc_score(y, oof_logreg_naive)
auc_rf_naive = roc_auc_score(y, oof_rf_naive)

print('BEFORE -- naive random KFold(5), a client can appear in both train and test:')
print(f'  AUC -- logistic regression : {auc_logreg_naive:.3f}')
print(f'  AUC -- random forest       : {auc_rf_naive:.3f}')

BEFORE -- naive random KFold(5), a client can appear in both train and test:
  AUC -- logistic regression : 0.742
  AUC -- random forest       : 0.771


### 2.2 After — the client-grouped split (Week 5's actual split)

Same models, same features, same label — `GroupKFold` by `client_hash_id` only, so a client's pages are entirely in one fold or the other. This is the split `w05_model.ipynb` reported as its headline number.

In [20]:
grouped_kfold = GroupKFold(n_splits=5)

oof_logreg_grouped = cross_val_predict(logreg, X, y, cv=grouped_kfold, groups=groups, method='predict_proba')[:, 1]
oof_rf_grouped = cross_val_predict(rf, X, y, cv=grouped_kfold, groups=groups, method='predict_proba')[:, 1]

auc_logreg_grouped = roc_auc_score(y, oof_logreg_grouped)
auc_rf_grouped = roc_auc_score(y, oof_rf_grouped)

print('AFTER -- GroupKFold(5) by client_hash_id, no client split across train/test:')
print(f'  AUC -- logistic regression : {auc_logreg_grouped:.3f}')
print(f'  AUC -- random forest       : {auc_rf_grouped:.3f}')

AFTER -- GroupKFold(5) by client_hash_id, no client split across train/test:
  AUC -- logistic regression : 0.719
  AUC -- random forest       : 0.721


In [21]:
import pandas as pd

before_after = pd.DataFrame({
    'before_naive_kfold': [auc_logreg_naive, auc_rf_naive],
    'after_grouped_kfold': [auc_logreg_grouped, auc_rf_grouped],
}, index=['logistic_regression', 'random_forest'])
before_after['gap'] = before_after['before_naive_kfold'] - before_after['after_grouped_kfold']
print(before_after.round(3))

print()
gap = before_after['gap'].max()
if gap > 0.03:
    print(f'VERDICT: the naive split overstates AUC by {gap:+.3f} at its worst -- client leakage was real, and '
          f'the grouped number is the one that belongs in any claim about how well this generalizes.')
elif gap > 0.01:
    print(f'VERDICT: a small but real gap ({gap:+.3f}) -- worth reporting, though most of the signal survives '
          f'grouping.')
else:
    print(f'VERDICT: essentially no gap ({gap:+.3f}) -- the honest split does not change the number much here, '
          f'which is itself worth stating rather than assuming client leakage was inflating things.')

                     before_naive_kfold  after_grouped_kfold    gap
logistic_regression               0.742                0.719  0.022
random_forest                     0.771                0.721  0.050

VERDICT: the naive split overstates AUC by +0.050 at its worst -- client leakage was real, and the grouped number is the one that belongs in any claim about how well this generalizes.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Week 3 (`w03_data_contract.ipynb`, Section 3.6) deliberately planted `pct_change_mar_vs_feb` as bait and watched it get caught. Same taxonomy, run here against the actual feature set trained above: label-derived features, future/overlapping windows, and product-flag/decision-derived columns.

### 3.1 No label-derived or sibling columns in the features

In [22]:
forbidden = {
    'is_declining_proxy',      # the label itself
    'gsc_impressions_feb',     # the label's other half
    'pct_change_mar_vs_feb',   # the label's exact source quantity (w03's planted trap)
    'trend_direction', 'trend_pct',   # the starter-CSV equivalents from w02, in case they ever got merged back in
}
leak = forbidden & set(X.columns)
assert not leak, f'Leaked columns made it into the model matrix: {leak}'
print('OK: none of', sorted(forbidden), 'are in the {} feature columns.'.format(X.shape[1]))

OK: none of ['gsc_impressions_feb', 'is_declining_proxy', 'pct_change_mar_vs_feb', 'trend_direction', 'trend_pct'] are in the 14 feature columns.


### 3.2 No product flags / existing-system scores as features

`baseline_score`, `reason_code`, and `action_label` are FlyRank's/my own rule's OUTPUT (Week 4/7) — they encode a decision already made from the same underlying signals and must never be inputs, per the "decision-derived features" branch of the leakage taxonomy.

In [23]:
decision_derived = {'baseline_score', 'score', 'reason_code', 'action_label', 'is_eligible',
                    'is_stale', 'ctr_position_gap', 'peer_median_ctr'}
leak2 = decision_derived & set(X.columns)
assert not leak2, f'Rule-derived/decision columns made it into the model matrix: {leak2}'
print('OK: none of the baseline rule\'s own outputs made it back in as features.')

OK: none of the baseline rule's own outputs made it back in as features.


### 3.3 No future window, and IDs are context only

Every feature above is built from `month=2026-03` alone; `2026-02` is touched only to build the label, and nothing from `2026-04` onward is queried anywhere in this notebook. `client_hash_id` / `content_hash_id` are used purely as join and grouping keys.

In [24]:
assert MONTH == '2026-03', 'Feature window drifted off the mid-panel month.'
assert PREV_MONTH == '2026-02', 'Comparison window drifted -- it should only ever build the label.'
assert not {'client_hash_id', 'content_hash_id'} & set(X.columns), (
    'ID columns leaked into the model matrix as features -- they are join/group keys only.'
)
print(f'OK: feature window is {MONTH} only; {PREV_MONTH} used only to build is_declining_proxy.')
print('OK: client_hash_id / content_hash_id are not in the feature matrix.')

OK: feature window is 2026-03 only; 2026-02 used only to build is_declining_proxy.
OK: client_hash_id / content_hash_id are not in the feature matrix.


### 3.4 Split is grouped, base rate is printed, and the top feature isn't hiding a leak

Section 2 already confirmed the split is grouped by `client_hash_id`. What's left from the checklist: print the base rate next to the metric (never a bare number), and directly test the one feature Week 5's permutation importance flagged as dominant (`log_impressions_mar`) — train once with it, once without, and confirm the AUC drop is a normal loss of a strong-but-legitimate feature, not the taxonomy's "near-perfect score collapses to ~0.7" tell for a disguised label-derived column.

In [25]:
base_rate = float(np.mean(y))
print(f'Base rate (share of pages that are is_declining_proxy=1): {base_rate:.3f}')
print(f'AFTER (grouped) AUC -- random forest : {auc_rf_grouped:.3f}  (vs base rate {base_rate:.3f})')

Base rate (share of pages that are is_declining_proxy=1): 0.201
AFTER (grouped) AUC -- random forest : 0.721  (vs base rate 0.201)


In [26]:
# Attack-your-own-model test: WITH vs WITHOUT the suspect feature, same grouped split.
X_without_suspect = X.drop(columns=['log_impressions_mar'])

oof_rf_without = cross_val_predict(
    rf, X_without_suspect, y, cv=grouped_kfold, groups=groups, method='predict_proba'
)[:, 1]
auc_rf_without = roc_auc_score(y, oof_rf_without)

drop = auc_rf_grouped - auc_rf_without
print(f'AUC WITH log_impressions_mar    : {auc_rf_grouped:.3f}')
print(f'AUC WITHOUT log_impressions_mar : {auc_rf_without:.3f}')
print(f'Drop: {drop:+.3f}')
print()
if auc_rf_grouped > 0.95 and drop > 0.2:
    print('FLAG: near-perfect score that collapses hard when the feature is removed -- this matches the '
          'label-derived-feature confession pattern. Treat log_impressions_mar as suspect, not settled.')
else:
    print('OK: AUC is not near-perfect and the drop is a plausible loss of one strong, legitimate feature -- '
          'not the taxonomy\'s collapse signature for a disguised label-derived column. Matches Week 5\'s own '
          'reading (Section 4.1/4.4 there): a real, dominant, but legitimate feature, not a leak.')

AUC WITH log_impressions_mar    : 0.721
AUC WITHOUT log_impressions_mar : 0.672
Drop: +0.049

OK: AUC is not near-perfect and the drop is a plausible loss of one strong, legitimate feature -- not the taxonomy's collapse signature for a disguised label-derived column. Matches Week 5's own reading (Section 4.1/4.4 there): a real, dominant, but legitimate feature, not a leak.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from `w05_model.ipynb`, Section 1):**

> "...signal 'spread thinly across several weak, interacting signals at once' is precisely what a Random Forest can learn and a hand-written AND-gate can't."

That sentence claims the model succeeds *because* it learned an interaction the baseline structurally couldn't represent. Section 4.1 of that same notebook already complicates it: permutation importance put `log_impressions_mar` at roughly seven times the weight of the next feature — the model's real skill is concentrated in one dominant signal, not a rich web of interacting weak ones. The original sentence was true as a *justification for trying* Random Forest in Section 1 (written before training), but it overstates what Section 4 later found actually happened.

**Rewrite, in safe language:**

> A Random Forest was chosen over a hand-written rule because the task's signals correlate only weakly with the label individually, and a fixed AND-gate can't combine weak signals the way a tree ensemble can. On this data, the trained model's ranking is *observed* to lean heavily on one feature — March impression volume — with the other candidate signals contributing comparatively little (Section 4.1's permutation-importance table). Whether this generalizes to a genuinely multi-signal pattern on a different month or client mix is not something this single audit can establish; the claim here is decision-support for prioritizing a review queue, not a demonstrated causal story about *why* pages decline.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.